# SVM su RBF branduoliu

SVC minkšto tarpo optimizavimas; K(z,z')=exp(-gamma ||z-z'||²).

Šis sąsiuvinis naudoja bendrus `src` modulius ir tą pačią kolokviume numatytą įdėtinę 5 × 10 CV. Galutinė visų metodų palyginimo ataskaita kuriama viena `run_experiment.py` komanda. Paleisti projekto šakniniame kataloge.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from src.data import load_vehicle
from run_experiment import make_outer_splits, fit_model, metrics

cfg = yaml.safe_load(Path('configs/main.yaml').read_text(encoding='utf-8'))
X, y, metadata = load_vehicle(Path('data'))
print(metadata)

In [ ]:
method = 'svm'
rows = []
for split, (repeat, fold, train, test) in enumerate(make_outer_splits(X, y, cfg)):
    model, params = fit_model(method, X.iloc[train], y.iloc[train], cfg, cfg['seed'] + repeat * 100 + fold, jobs=1)
    score = metrics(y.iloc[test], model.predict(X.iloc[test]))
    rows.append({'split': split, 'repeat': repeat, 'fold': fold, **score, 'params': str(params)})
results = pd.DataFrame(rows)
results.head()

In [ ]:
display(results[['macro_f1', 'balanced_accuracy']].agg(['mean', 'std']))
Path('results').mkdir(exist_ok=True)
results.to_csv(f'results/notebook_{method}.csv', index=False)
results.boxplot(column=['macro_f1', 'balanced_accuracy'])
plt.title(f'{method}: 50 išorinių bandymų')
plt.show()

Metrikų skirtumus su baseline, abliacijas, atsparumą ir painiavos matricą pateikia pagrindinis eksperimentas. Šio sąsiuvinio rezultatai neturi būti atrenkami pagal išorinį bandymą hiperparametrams keisti.